In [1]:
import pandas as pd
from openai import OpenAI
from autoddg import AutoDDG
from autoddg.utils import get_sample
from autoddg.evaluation import BaseEvaluator
from typing import Optional
# --- Import custom files ---
from prompts import ALL_RELATED_WORK_PROMPTS
from utils import log_result 
import os 
import json

In [2]:
# --- LLM Config ---
MODEL_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    "model_name": "llama3.1:8b",
}

from cache_utils import run_with_caching, load_profile_from_cache, MockAutoDDG 
# You will likely replace MockAutoDDG with your actual AutoDDG class import

# 1. Instantiate your dependencies
profiling_engine = MockAutoDDG() # Or your actual AutoDDG()

#single 
# --- Experiment Config ---
DATABASE_PATH_ = '../src/autoddg/database.json'  # Ensure this path is correct\n",
RESULTS_FILE = 'prompt-experiments/autoddg_experiment_results.csv'
PROFILE_CACHE_DIR = 'profile_cache' # Directory to save/load profiles\n",

script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
DATABASE_PATH = os.path.join(script_dir, DATABASE_PATH_)
PROJECT_ROOT = os.path.abspath(os.path.join(script_dir, os.pardir))
# ABSOLUTE_CACHE_DIR = os.path.join(script_dir, PROFILE_CACHE_DIR)


# --- Define Evaluation Class ---
class Eval(BaseEvaluator):
    def __init__(self, model_name: str = MODEL_CONFIG["model_name"]):
        client = OpenAI(
            api_key=MODEL_CONFIG["api_key"], 
            base_url=MODEL_CONFIG["base_url"]
        )
        super().__init__(client=client, model_name=model_name)

# Initialize Core Tools
client = OpenAI(api_key=MODEL_CONFIG["api_key"], base_url=MODEL_CONFIG["base_url"])
auto_ddg = AutoDDG(client=client, model_name=MODEL_CONFIG["model_name"])
auto_ddg.set_evaluator(Eval())

[SETUP] Project Root: /home/bia/Documents/AutoDDG-Enhanced
[SETUP] Cache Directory: /home/bia/Documents/AutoDDG-Enhanced/prompt-experiments/profile_cache


In [3]:
database = None
try:
    # Load the database JSON
    with open(DATABASE_PATH, 'r') as f:
        database = json.load(f)
        print(database)
except FileNotFoundError:
    print(f"ERROR: Database file not found at {DATABASE_PATH}")
except json.JSONDecodeError:
    print(f"ERROR: Could not decode JSON from {DATABASE_PATH}. Is the file correctly formatted?")
except Exception as e:
    print(f"An unexpected error occurred while loading the database: {e}")

# Iterate through the database entries
for dataset_id, dataset_info in database.items():
    try:
        # This calls the function that implements the cache-first logic
        run_with_caching(dataset_id, dataset_info, profiling_engine) 
    except Exception as e:
        print(f"\n{'#'*50}")
        print(f"FATAL ERROR: FAILED ON DATASET ID {dataset_id} ({dataset_info.get('dataset_name', 'Unknown')})")
        print(f"Error: {e}")
        print(f"{'#'*50}\n")
        # Continue to the next dataset instead of stopping the whole loop
        continue

print("\n=======================================================")
print("ALL CACHING RUNS FINISHED.")
print("=======================================================")


{'3222451': {'dataset_name': 'The FluPRINT database', 'dataset_path': 'src/autoddg/related/data/fluprint_export.csv', 'related_paper_path': 'src/autoddg/related/papers/3222451.pdf', 'description': 'The FluPRINT represents fully integrated and normalized immunology measurements from eight clinical studies taken from 740 individuals undergoing influenza vaccination with inactivated or live attenuated seasonal influenza vaccines from 2007 to 2015 at the Stanford Human Immune Monitoring Center. The FluPRINT dataset contains information on more than 3,000 parameters measured using mass cytometry, flow cytometry, phosphorylation-specific cytometry, multiplex cytokine assays, clinical lab tests (hormones and complete blood count), serological profiling and virological tests. In the dataset, vaccine protection is measured using a hemagglutination inhibition (HAI) assay, and following FDA guidelines individuals are marked as high or low responders depending on the HAI antibody titers after vacc

In [4]:
test_id = '4296944'
test_profile = load_profile_from_cache(test_id)

[CACHE] Loaded profiles for 4296944 from cache.


In [5]:
print(test_profile)

{'basic_profile': {'col_count': 132, 'df_info': 'mocked'}, 'structural_profile': {'row_count': 12285, 'missing': 0}, 'semantic_profile': {'semantic_score': 0.85, 'sample_analysis': 'mocked'}, 'data_topic': 'Auto-Generated Topic: FCCdb: A large-scale manually curated database of food chemical compositions', 'dataset_sample': ['Sample data point 1', 'Sample data point 2']}


In [6]:
print(test_profile['semantic_profile'])

{'semantic_score': 0.85, 'sample_analysis': 'mocked'}


In [29]:
# Cell 3: Data Loading and Core Profiling (Run Once)

print("--- Loading Data and Running Core Profiling ---")
df = pd.read_csv(DATA_FILE)
sample_df, dataset_sample = get_sample(df, sample_size=100)

basic_profile, structural_profile = auto_ddg.profile_dataframe(df)
semantic_profile = auto_ddg.analyze_semantics(sample_df)
data_topic = auto_ddg.generate_topic(DATASET_NAME, None, dataset_sample)

print("Profiling Complete.")

--- Loading Data and Running Core Profiling ---


NameError: name 'DATA_FILE' is not defined

In [ ]:
def run_with_caching(dataset_id, dataset_info):
    """Demonstrates the core cache-first logic."""
    print(f"\n[RUNNER] Attempting to process {dataset_info['dataset_name']}...")
    
    # 1. Try to load from cache
    profiles = load_profile_from_cache(dataset_id)
    
    if profiles is None:
        # 2. If load fails, generate and cache
        profiles = generate_and_cache_profiles(dataset_info, dataset_id, auto_ddg)
        
    if profiles:
        print(f"[RUNNER] Successfully retrieved profiles for {dataset_info['dataset_name']}.")
        # Example of using the retrieved profile
        print(f"  Topic: {profiles['data_topic']}")
    else:
        print(f"[RUNNER] Failed to get profiles for {dataset_info['dataset_name']}.")


In [12]:
# Cell 4: Define Prompts to Test (Selects from the imported dictionary)

# Define which prompts you want to run for this experiment
PROMPTS_TO_TEST = {
    "V1_Revised": ALL_RELATED_WORK_PROMPTS["V1_Revised"],
    # "V2_Aggressive": ALL_RELATED_WORK_PROMPTS["V2_Aggressive"],
    "V2_Hybrid": ALL_RELATED_WORK_PROMPTS["V2_Hybrid"]
}
print(f"Testing {len(PROMPTS_TO_TEST)} related work prompts.")



Testing 2 related work prompts.


In [ ]:
# Cell 5: Run and Log Baseline (Vanilla) Description

print("\n--- Running Baseline (Vanilla) Test ---")
prompt_baseline, description_baseline = auto_ddg.describe_dataset(
        dataset_sample=dataset_sample,
        dataset_profile=basic_profile,
        use_profile=True,
        semantic_profile=semantic_profile,
        use_semantic_profile=True,
        data_topic=data_topic,
        use_topic=True,
        use_related_profile=False  # Vanilla
    )

baseline_scores = auto_ddg.evaluate_description(description_baseline)
print(f"Baseline Scores: {baseline_scores}")

# Log result using the new utility function (passing required file/dataset args)
log_result(
    prompt_name="N/A", 
    description_type="Vanilla_AutoDDG", 
    description=description_baseline, 
    raw_scores=baseline_scores,
    dataset_name=DATASET_NAME,
    file_path=RESULTS_FILE
    # related_profile is omitted (defaults to None)
)


--- Running Baseline (Vanilla) Test ---


TypeError: AutoDDG.describe_dataset() got an unexpected keyword argument 'use_related_profile'

In [7]:
# Cell 6: Run and Log Augmented Descriptions for Multiple Prompts

for prompt_name, extraction_prompt in PROMPTS_TO_TEST.items():
    print(f"\n--- Running Augmented Test with Prompt: {prompt_name} ---")
    
    # Step A: Analyze related work using the current prompt
    related_profile = auto_ddg.analyze_related(
        pdf_path=PAPER_FILE,
        dataset_name=DATASET_NAME,
        extraction_prompt=extraction_prompt,
        max_pages=10
    )
    print(f"Related Work Summary: {related_profile['summary'][:150]}...")

    # Step B: Generate description with the new related profile
    prompt_augmented, description_augmented = auto_ddg.describe_dataset(
        dataset_sample=dataset_sample,
        dataset_profile=basic_profile,
        use_profile=True,
        semantic_profile=semantic_profile,
        use_semantic_profile=True,
        data_topic=data_topic,
        use_topic=True,
        related_profile=related_profile,
        use_related_profile=True # Augmented
    )
    
    # Step C: Evaluate and Log
    augmented_scores = auto_ddg.evaluate_description(description_augmented)
    print(f"Augmented Scores ({prompt_name}): {augmented_scores}")
    
    log_result(
        prompt_name=prompt_name, 
        description_type="Augmented_AutoDDG", 
        description=description_augmented, 
        raw_scores=augmented_scores,
        dataset_name=DATASET_NAME,
        file_path=RESULTS_FILE,
        related_profile=related_profile # Now logs the profile!
    )
    
print("\nAll experiments complete. Results saved to:", RESULTS_FILE)

Ignoring wrong pointing object 43 0 (offset 0)



--- Running Augmented Test with Prompt: V1_Revised ---
Reading PDF from: ../src/autoddg/related/papers/code15.pdf
Successfully extracted text from 10 pages (total: 10 pages)
Total characters extracted: 55673
Extracting related work profile for dataset: CODE-15%: a large scale annotated dataset of 12-lead ECGs
Sending 56479 characters to LLM...
Successfully extracted profile (689 characters)
Related Work Summary: The dataset search engine has extracted the following factual context:

* The study found that deep learning models can accurately predict atrial fibr...


Ignoring wrong pointing object 43 0 (offset 0)


Augmented Scores (V1_Revised): Here are my scores based on the Evaluation Criteria:

Evaluation Form (scores ONLY):

Completeness: 9
Conciseness: 8
Readability: 9
Logged Augmented_AutoDDG with Prompt V1_Revised to autoddg_experiment_results.csv

--- Running Augmented Test with Prompt: V2_Hybrid ---
Reading PDF from: ../src/autoddg/related/papers/code15.pdf
Successfully extracted text from 10 pages (total: 10 pages)
Total characters extracted: 55673
Extracting related work profile for dataset: CODE-15%: a large scale annotated dataset of 12-lead ECGs
Sending 56621 characters to LLM...
Successfully extracted profile (2333 characters)
Related Work Summary: **Dataset Entry:**

**Title:** Comparative Analysis of Deep Learning Models for Electrocardiogram (ECG) Signal Processing and Arrhythmia Detection

**...
Augmented Scores (V2_Hybrid): Here are my scores based on the Evaluation Criteria:

Completeness: 9
Conciseness: 8
Readability: 9
Logged Augmented_AutoDDG with Prompt V2_Hybrid to auto

In [ ]:

import json
import os

def run_experiment(dataset):
    
    
    dataset_name = dataset["dataset_name"]
    data_file = dataset["dataset_path"]
    paper_file = dataset.get("related_paper_path")
        
    print("--- Loading Data and Running Core Profiling ---")
    #make sure size is appropriate
    if os.path.getsize(data_file) > 10 * 1024 * 1024:  # 10 MB size limit
        #sample 
        df = pd.read_csv(data_file, nrows=100000)  
        
    else: 
        df = pd.read_csv(data_file)
    sample_df, dataset_sample = get_sample(df, sample_size=100)

    basic_profile, structural_profile = auto_ddg.profile_dataframe(df)
    semantic_profile = auto_ddg.analyze_semantics(sample_df)
    data_topic = auto_ddg.generate_topic(DATASET_NAME, None, dataset_sample)

    print("Profiling Complete.")
    
        
    print("\n--- Running Baseline (Vanilla) Test ---")
    prompt_baseline, description_baseline = auto_ddg.describe_dataset(
            dataset_sample=dataset_sample,
            dataset_profile=basic_profile,
            use_profile=True,
            semantic_profile=semantic_profile,
            use_semantic_profile=True,
            data_topic=data_topic,
            use_topic=True,
            use_related_profile=False  # Vanilla
        )

    baseline_scores = auto_ddg.evaluate_description(description_baseline)
    print(f"Baseline Scores: {baseline_scores}")

    # Log result using the new utility function (passing required file/dataset args)
    log_result(
        prompt_name="N/A", 
        description_type="Vanilla_AutoDDG", 
        description=description_baseline, 
        raw_scores=baseline_scores,
        dataset_name=DATASET_NAME,
        file_path=RESULTS_FILE
        # related_profile is omitted (defaults to None)
    )
    
    PROMPTS_TO_TEST = {
        "V1_Revised": ALL_RELATED_WORK_PROMPTS["V1_Revised"],
        "V2_Hybrid": ALL_RELATED_WORK_PROMPTS["V2_Hybrid"]
    }
    
        
    for prompt_name, extraction_prompt in PROMPTS_TO_TEST.items():
        print(f"\n--- Running Augmented Test with Prompt: {prompt_name} ---")
        
        # Step A: Analyze related work using the current prompt
        related_profile = auto_ddg.analyze_related(
            pdf_path=paper_file,
            dataset_name=dataset_name,
            extraction_prompt=extraction_prompt,
            max_pages=10
        )
        print(f"Related Work Summary: {related_profile['summary'][:150]}...")

        # Step B: Generate description with the new related profile
        prompt_augmented, description_augmented = auto_ddg.describe_dataset(
            dataset_sample=dataset_sample,
            dataset_profile=basic_profile,
            use_profile=True,
            semantic_profile=semantic_profile,
            use_semantic_profile=True,
            data_topic=data_topic,
            use_topic=True,
            related_profile=related_profile,
            use_related_profile=True # Augmented
        )
        
        # Step C: Evaluate and Log
        augmented_scores = auto_ddg.evaluate_description(description_augmented)
        print(f"Augmented Scores ({prompt_name}): {augmented_scores}")
        
        log_result(
            prompt_name=prompt_name, 
            description_type="Augmented_AutoDDG", 
            description=description_augmented, 
            raw_scores=augmented_scores,
            dataset_name=DATASET_NAME,
            file_path=RESULTS_FILE,
            related_profile=related_profile # Now logs the profile!
        )
        
    print("\nDataset {dataset} completed.--> Results saved to:", RESULTS_FILE)


In [ ]:
       
    
###Runnning all datasets in the database

#Load the dabase 
with open("src/autoddg/database.json") as f:
    database = json.load(f)



for dataset_id, dataset in database.items():
    try:
        run_experiment(dataset)
    except Exception as e:
        print(f"FAILED ON {dataset_id}: {e}")
        continue

In [ ]:


#LOOP THROUGH ALL DATASETS IN DATABASE

# --- Experiment Config ---
RESULTS_FILE = "autoddg_experiment_results.csv"

# --- Load Database ---
with open("src/autoddg/database.json") as f:
    database = json.load(f)

# --- Iterate through datasets ---
for dataset_id, dataset in database.items():
    DATASET_NAME = dataset["dataset_name"]
    DATA_FILE = dataset["dataset_path"]
    PAPER_FILE = dataset.get("related_paper_path")

    print("Running experiment on:", DATASET_NAME)
    print("Data file:", DATA_FILE)
    print("Paper file:", PAPER_FILE)
    print("----")
    
    # Run Evaluation
    print("--- Loading Data and Running Core Profiling ---")
    
    #some datasets are too large to load fully and will have to be sampled 
    if os.path.getsize(DATA_FILE) > 100000000:  # 100 MB size limit for full load
        print("Dataset too large, loading a sample for profiling.")
        df = pd.read_csv(DATA_FILE, nrows=10000)  # Load only first 10,000 rows for sampling
    else: 
        df = pd.read_csv(DATA_FILE)
        
    sample_df, dataset_sample = get_sample(df, sample_size=100)

    basic_profile, structural_profile = auto_ddg.profile_dataframe(df)
    semantic_profile = auto_ddg.analyze_semantics(sample_df)
    data_topic = auto_ddg.generate_topic(DATASET_NAME, None, dataset_sample)

    print("Profiling Complete.")
    